### Standard Scaler

`StandardScaler` standardizes each numeric feature by removing its mean and scaling to unit variance (media ≈ 0, std ≈ 1)
**When to use**
- Features are on very different scales.
- Models are distance- or gradient-based (e.g., KNN, SVM, logistic/linear regression, neural networks, PCA).
- You want faster, more stable optimization and comparable feature magnitudes.

It is usually **not necessary** for tree-based models (Decision Trees, Random Forest, Gradient Boosting).

> **Source:** scikit-learn User Guide — [7.3.1 Standardization, or mean removal and variance scaling](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling)

##### Quick usage (scikit-learn)

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
```

- Fit on training data only.
- Use the same fitted scaler to transform validation/test data.

### Min-Max Scaler

`MinMaxScaler` rescales each numeric feature to a fixed range (default $[0,1]$)

**When to use**
- You need features bounded to the same interval (commonly $[0,1]$).
- Models or pipelines benefit from fixed input ranges (e.g., neural networks, distance-based methods).
- You want to preserve the original shape/order of values while changing scale.

It can be sensitive to outliers, since extreme values set the min and max.

> **Source:** scikit-learn User Guide — [7.3.1.1 Scaling features to a range](https://scikit-learn.org/stable/modules/preprocessing.html#scaling-features-to-a-range)

##### Quick usage (scikit-learn)

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_m = scaler.fit_transform(X_train)
X_test_m = scaler.transform(X_test)
```

- Fit on training data only.
- Use the same fitted scaler to transform validation/test data.

### Logarithmic Normalization

Log normalization compresses large values and reduces right-skew in long-tail distributions.

$$
\text{Natural log (base }e\text{):}\quad x' = \ln(x)
$$

$$
\text{Zero-safe version:}\quad x' = \ln(1+x)=\operatorname{log1p}(x)
$$

**When to use**
- Features are positive and highly right-skewed (e.g., income, counts, sales).
- Very large values dominate model behavior.
- You want to stabilize variance before applying scaling (often `log1p` then `StandardScaler`).

`\ln(x)` requires $x>0$, while `log1p(x)` also works when $x=0$.

> **Source:** NumPy docs — [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html); Géron, A. *Hands-On Machine Learning*, O'Reilly, Ch. 2 (feature engineering heuristics).

##### Quick usage (NumPy)

```python
import numpy as np

X_train_log = np.log1p(X_train)
X_test_log = np.log1p(X_test)
```

- Apply the same transformation to train/validation/test.
- If negatives exist, shift or use an alternative transform first.

### LabelEncoder

`LabelEncoder` converts categorical target labels into integer values (e.g., `"bad" -> 0`, `"good" -> 1`) so models can work with them.

**When to use**
- Your target variable `y` is categorical text labels.
- You need numeric class IDs for classification workflows.
- You want to map predictions back to original class names after inference.

**Limitation**
- Use it for the target (`y`), not for nominal input features (`X`) with no natural order.

> **Source:** scikit-learn User Guide — [7.3.4 Encoding categorical features](https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features)

##### Quick usage (scikit-learn)

```python
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Optional: recover original labels
# y_pred_labels = le.inverse_transform(y_pred_enc)
```

- Fit on training labels only.
- Reuse the same encoder for validation/test labels.

For categorical input features in `X`, prefer `OneHotEncoder`.

```python
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat_train_ohe = ohe.fit_transform(X_cat_train)
X_cat_test_ohe = ohe.transform(X_cat_test)
```



### Grid Search CV

`GridSearchCV` is a systematic way to find the best hyperparameters using cross-validation: it tries all parameter combinations and keeps the one with the best average validation score.

**When to use**
- You have a small/medium hyperparameter space and want a reliable baseline.
- You want a more objective selection than manual trial-and-error.
- You need stronger generalization estimates via cross-validation.

**Limitation**
- It can become computationally expensive when the parameter grid is large.

> **Source:** scikit-learn User Guide — [3.2 Tuning the hyper-parameters of an estimator](https://scikit-learn.org/stable/modules/grid_search.html)

##### Quick usage (scikit-learn)

```python
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
best_params = grid.best_params_
```

- `cv=3`: 3-fold cross-validation.
- `n_jobs=-1`: uses all CPU cores.
- `verbose=2`: prints detailed progress during search.

### PCA (Principal Component Analysis)

`PCA` is a dimensionality reduction method that transforms the original features into new uncorrelated components, ordered by how much variance they explain.

$$
Z = XW
$$

Where $W$ contains the principal directions (eigenvectors), and the first components keep most of the dataset information.

**When to use**
- You have many numeric features with strong correlation.
- You want to reduce dimensionality while preserving most variance.
- You need faster training or less noise before modeling.

**Limitation**
- Components are harder to interpret because they are linear combinations of original features.

> **Source:** scikit-learn User Guide — [2.5 Decomposing signals in components (PCA)](https://scikit-learn.org/stable/modules/decomposition.html#pca); Goodfellow, I., Bengio, Y. & Courville, A. *Deep Learning*, MIT Press (2016), Ch. 2.12.

##### Quick usage (scikit-learn)

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=0.90)  # keep ~90% explained variance
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
```

- Fit `PCA` on training data only.
- Apply the same fitted transform to validation/test sets.
- Standardize features before PCA in most cases.

### Decision Tree

A Decision Tree recursively partitions the feature space by choosing the split at each node that maximally reduces impurity (e.g., Gini impurity or entropy). Each leaf node assigns a class label (classification) or a value (regression).

**When to use**
- You need an interpretable model (rules can be visualized).
- Features are a mix of numeric and categorical types.
- Scaling is not required — trees are invariant to monotonic feature transformations.

**Limitation**
- Prone to overfitting when depth is unconstrained; use `max_depth` or pruning.
- High variance: small data changes can produce very different trees.

> **Source:** scikit-learn User Guide — [1.10 Decision Trees](https://scikit-learn.org/stable/modules/tree.html); Breiman, L. et al. *Classification and Regression Trees*, Wadsworth (1984).

##### Quick usage (scikit-learn)

```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

print(confusion_matrix(y_test, y_pred))
```

- `max_depth` limits tree growth and reduces overfitting.
- Use `feature_importances_` to inspect which features drive predictions.

### KNN (K-Nearest Neighbors)

KNN classifies a new sample by majority vote among the $k$ closest training points in feature space.

**When to use**
- Simple baseline for classification or regression.
- Decision boundaries are naturally non-linear.
- Dataset is small to medium (KNN is lazy — no explicit training, but slow at inference on large data).

**Limitation**
- Sensitive to feature scale: always standardize before using KNN.
- Slow prediction on large datasets ($O(n)$ per query).
- Performance degrades with irrelevant features (curse of dimensionality).

> **Source:** scikit-learn User Guide — [1.6 Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html); Cover, T. & Hart, P. "Nearest neighbor pattern classification", *IEEE Transactions on Information Theory* 13(1), 1967.

##### Quick usage (scikit-learn)

```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred = knn.predict(X_test_s)
```

- Tune `n_neighbors` via cross-validation — smaller $k$ = more complex boundary, larger $k$ = smoother.
- Always scale features before fitting KNN.

### Random Forest

Random Forest is an ensemble of Decision Trees trained on random bootstrap samples of the data, each tree also selecting a random subset of features at every split. Final prediction is the majority vote (classification) or average (regression) across all trees.

**When to use**
- You need a robust, high-accuracy baseline with low risk of overfitting.
- Dataset has mixed feature types or noisy features.
- You want built-in feature importance scores.

**Limitation**
- Less interpretable than a single Decision Tree.
- Slower to train and predict than a single tree, especially with many estimators.

> **Source:** scikit-learn User Guide — [1.11 Ensembles: Random Forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests); Breiman, L. "Random Forests", *Machine Learning* 45(1), 5–32, 2001.

##### Quick usage (scikit-learn)

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

# Feature importances
import pandas as pd
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
```

- No need to scale features — tree-based.
- Key hyperparameters: `n_estimators`, `max_depth`, `min_samples_split` (tune with `GridSearchCV`).

### Cross-Validation

Cross-validation (CV) estimates how well a model generalizes to unseen data by repeatedly splitting the data into training and validation folds. The most common variant is **k-fold CV**: the dataset is divided into $k$ equal folds; the model trains on $k-1$ folds and validates on the remaining one, cycling through all folds.

**When to use**
- You want a more reliable generalization estimate than a single train/test split.
- Comparing multiple models or hyperparameter configurations.
- Dataset is small and a fixed test split would waste too much data.

**Limitation**
- Computationally more expensive than a single split ($k$ times more training runs).
- Standard k-fold may produce unbalanced class distributions per fold — use `StratifiedKFold` for classification.

> **Source:** scikit-learn User Guide — [3.1 Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html); Hastie, T., Tibshirani, R. & Friedman, J. *The Elements of Statistical Learning*, 2nd ed., Springer (2009), §7.10.

##### Quick usage (scikit-learn)

```python
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy', n_jobs=-1)

print(f"Mean accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
```

- `cv=5`: 5-fold cross-validation.
- Use `StratifiedKFold` explicitly when class imbalance matters.
- `GridSearchCV` uses cross-validation internally for hyperparameter search.

### Feature Selection

Feature selection chooses the most informative subset of input features to keep, discarding redundant or noisy ones. Unlike PCA, it does not transform features — it selects from the originals.

Common strategies:
- **Filter methods**: rank features independently of the model (e.g., correlation, variance threshold).
- **Wrapper methods**: use model performance to score feature subsets (e.g., RFE — Recursive Feature Elimination).
- **Embedded methods**: selection happens during model training (e.g., `feature_importances_` in tree-based models, L1 regularization in linear models).

**When to use**
- Dataset has many features and some are likely irrelevant or redundant.
- You want to reduce overfitting, speed up training, or improve interpretability.
- Model performance degrades with too many features (curse of dimensionality for KNN, etc.).

**Limitation**
- Wrapper methods (RFE) are computationally expensive on large feature sets.
- Filter methods ignore feature interactions.

> **Source:** scikit-learn User Guide — [1.13 Feature selection](https://scikit-learn.org/stable/modules/feature_selection.html); Guyon, I. & Elisseeff, A. "An Introduction to Variable and Feature Selection", *JMLR* 3, 2003.

##### Quick usage (scikit-learn)

```python
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=10)
rfe.fit(X_train, y_train)

X_train_sel = rfe.transform(X_train)
X_test_sel = rfe.transform(X_test)

# See which features were selected
selected = [name for name, s in zip(feature_names, rfe.support_) if s]
```

- Alternatively, use `feature_importances_` from a trained Random Forest to rank and drop low-importance features.
- Always apply feature selection only on training data, then transform validation/test.